# F2 · Unión de las tres tablas, auditoría del join y holdout dev/test

Este notebook hace tres cosas, **en este orden y por este motivo**:

1. **Une** las tres tablas crudas por `VehicleCode` —a nivel vehículo, y
   enriqueciendo `trips` y `signals` por separado, que son las dos uniones que el
   dato admite— aplicando el dedupe de los 13 vehículos clonados
   (`src/data/dedupe.py`) antes de mirar un solo número.
2. **Audita** esquema y calidad sobre esa unión —nulos, tipos, duplicados,
   cobertura del join— y re-confirma los hallazgos de F1 contra lo documentado en
   [`docs/memoria/`](../docs/memoria/README.md).
3. **Congela** el split dev/test 80/20, agrupado por vehículo y estratificado por
   `event_observed`.

El orden no es casual. La auditoría va primero porque **no depende de qué vehículo
caiga de cada lado**: contar nulos o verificar que el join cierra da el mismo
resultado con cualquier partición, así que hacerla sobre el dataset completo no
contamina nada. Todo lo que sí depende de los datos —elegir features, fijar W/G/H,
mover un umbral— va *después* del split y *solo* sobre dev. Si el test se eligiera
al final, cada una de esas decisiones se habría tomado mirándolo, y el número que
reportemos en el pitch sería optimista sin que nadie pueda decir por cuánto.

**Este notebook no tiene lógica de negocio.** La unión vive en `src/data/join.py`,
el dedupe en `src/data/dedupe.py` y el split en `src/eval/splits.py`; acá solo se
ejecutan y se miran. Los parámetros (semilla, `test_size`, paths) salen de
`configs/data/test_split.yaml`, no están escritos en ninguna celda.

Reproducir sin notebook (misma semilla, mismo resultado):

```bash
python scripts/make_test_split.py --config configs/data/test_split.yaml
```

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from src.config import load_config, resolve_path, set_seed
from src.data.join import (
    anchor_offset_days,
    build_vehicle_table,
    duplicate_report,
    enrichment_report,
    iter_enriched_table,
    join_coverage,
    load_vehicle_static,
    quality_report,
    scan_raw_tables,
)
from src.data.loader import load_table
from src.eval.splits import (
    load_test_split,
    make_test_split,
    save_test_split,
    split_balance,
    test_split_masks,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 120)

CONFIG = "configs/data/test_split.yaml"
cfg = load_config(CONFIG)
set_seed(int(cfg["seed"]))
cfg

{'seed': 42,
 'test_size': 0.2,
 'sources': 'configs/data/raw_sources.yaml',
 'dedupe': 'configs/data/vehicle_dedupe.yaml',
 'panel': 'configs/data/panel_v1.yaml',
 'group_column': 'vehicle_id',
 'event_column': 'event_observed',
 'balance_columns': ['static_Engine',
  'static_ModelSeries',
  'static_SalesCountry_cd'],
 'scan': {'chunksize': 500000},
 'output': {'test_split': '${DATA_DIR}/processed/test_split.json',
  'vehicle_table': '${DATA_DIR}/processed/vehicles.parquet',
  'quality': '${DATA_DIR}/processed/raw_quality.csv'},
 '_config_path': 'configs\\data\\test_split.yaml'}

## 1 · Carga, dedupe y unión

`vehicles` tiene una fila por vehículo, `trips` una por viaje (2,53M) y `signals`
una por mensaje del postratamiento (10,6M). Las tres comparten `VehicleCode`,
pero **`trips` y `signals` no se mergean entre sí**: no hay clave fila a fila
entre ellas, así que un join por vehículo es muchos-a-muchos y devuelve el
producto de los dos históricos. Para el vehículo mediano son 1971 viajes × 7195
señales ≈ 14,2M de filas de un solo vehículo; sobre los 1081, 3,2×10¹⁰ filas. No
es que tarde: no entra. Alinear señales contra viajes se hace por
odómetro/timestamp y es trabajo de F2 (`trips` manda, ver
[`f1-calidad-odometro.md`](../docs/memoria/f1-calidad-odometro.md)).

Las dos uniones que sí corresponden:

- **Enriquecer cada tabla grande por separado** con las estáticas del vehículo
  (§1.2): left join 1-a-muchos, cada tabla conserva sus filas, dos tablas
  separadas.
- **Bajar a nivel vehículo** (§1.1): una fila por `vehicle_id`. Es lo único que
  hace falta para auditar el join y congelar el holdout.

### 1.1 · Nivel vehículo

`scan_raw_tables` hace **una sola pasada por tabla** y en esa pasada cuenta nulos
por columna, detecta duplicados por hash de fila y arma los agregados por
vehículo. El dedupe se aplica adentro, antes de cualquier agregado: si no, los 13
clones aparecerían con el cuádruple de viajes.

In [2]:
scans = scan_raw_tables(
    config_path=cfg["sources"],
    dedupe_config=cfg["dedupe"],
    chunksize=int(cfg["scan"]["chunksize"]),
)
static = load_vehicle_static(
    config_path=cfg["sources"], dedupe_config=cfg["dedupe"], panel_config=cfg["panel"]
)
vehicles = build_vehicle_table(static, scans)

print({name: scan.n_rows for name, scan in scans.items()})
print("vehículos tras el dedupe:", len(vehicles))
vehicles.head(3)

{'vehicles': 1120, 'trips': 2531435, 'signals': 10617817}
vehículos tras el dedupe: 1081


,vehicle_id,cohort,event_observed,event_day_since_production,static_ModelSeries,static_SalesCountry_cd,static_daysUntilSale,static_ProductionDay,static_Engine,n_trips,...,trip_date_max,trip_km_negative,in_trips,n_signals,signal_odo_null,signal_odo_max,signal_date_min,signal_date_max,n_regenerations,in_signals
0,VEH_0001,not_failed,0,NaN,MODEL_4,CNTRY_1,47.0,59.0,ENG_1,5983,...,2026-09-13 23:20:44+00:00,0,1,18687,0,36381.0,2025-03-21 11:39:59+00:00,2026-09-12 20:18:56+00:00,188,1
1,VEH_0002,not_failed,0,NaN,MODEL_4,CNTRY_1,45.0,54.0,ENG_1,82,...,2025-05-05 22:57:05+00:00,0,1,138,1,39.0,2025-03-18 09:43:52+00:00,2025-05-05 23:15:15+00:00,0,1
2,VEH_0003,not_failed,0,NaN,MODEL_4,CNTRY_1,110.0,92.0,ENG_1,2017,...,2026-09-13 19:01:14+00:00,0,1,6856,0,10244.0,2025-04-23 09:16:08+00:00,2026-09-12 19:19:48+00:00,31,1


### 1.2 · `trips` y `signals` enriquecidas, cada una por su lado

`iter_enriched_table` pega las estáticas del vehículo a cada tabla grande con un
left join 1-a-muchos, por chunks. La tabla sale con **las mismas filas** y solo
columnas nuevas: es un enriquecimiento, no una fusión.

El único lugar donde cambia el total es el colapso de los clones, y ahí cambia a
propósito: sus copias están en archivos distintos (una en cada cohorte), así que
no caen nunca en el mismo chunk y hay que retenerlas y deduplicarlas enteras al
final. Por eso `filas_esperadas = filas_crudas − dup_clones`.

`huerfanas` es la validación fila a fila de que `VehicleCode` cierra: cuenta filas
cuyo vehículo no existe en `vehicles`. Tiene que dar cero en las dos tablas.

In [3]:
enriquecidas = enrichment_report(
    static,
    scans,
    config_path=cfg["sources"],
    dedupe_config=cfg["dedupe"],
    chunksize=int(cfg["scan"]["chunksize"]),
)
enriquecidas

,table,filas_crudas,filas_esperadas,filas_enriquecidas,huerfanas,columnas
0,trips,2531435,2438339,2438339,0,26 -> 33
1,signals,10617817,10232950,10232950,0,8 -> 15


In [4]:
# Cómo queda una de las dos (primer chunk de `trips`): mismas filas, columnas nuevas al final.
chunk = next(
    iter_enriched_table("trips", static, config_path=cfg["sources"], dedupe_config=cfg["dedupe"], chunksize=50_000)
)
print("columnas pegadas:", [c for c in chunk.columns if c.startswith("static_") or c.startswith("event_")])
chunk.head(3)

columnas pegadas: ['event_observed', 'event_day_since_production', 'static_ModelSeries', 'static_SalesCountry_cd', 'static_daysUntilSale', 'static_ProductionDay', 'static_Engine']


,vehicle_id,TripDatetimeStart,TripDatetimeEnd,OdometerTripStart,OdometerTripEnd,KilometerPerHour,FuelLvlStartPc,FuelLvlEndPc,EngineOilLifePCStart,EngineOilLifePCEnd,...,AirTemperatureAvg,TripNumber,cohort,event_observed,event_day_since_production,static_ModelSeries,static_SalesCountry_cd,static_daysUntilSale,static_ProductionDay,static_Engine
0,VEH_0260,2026-08-13 23:39:18+00:00,2026-08-13 23:48:48+00:00,13515.0,13518.0,18.9474,103.0438,102.8264,0.0,0.0,...,-0.4167,1636.0,failed,1,174.0,MODEL_4,CNTRY_1,174.0,128.0,ENG_1
1,VEH_0260,2026-08-10 22:42:26+00:00,2026-08-10 22:57:01+00:00,13453.0,13457.0,16.4571,17.3914,16.8479,0.0,0.0,...,4.9167,1618.0,failed,1,174.0,MODEL_4,CNTRY_1,174.0,128.0,ENG_1
2,VEH_0641,2026-08-05 14:52:29+00:00,2026-08-05 14:53:12+00:00,57630.0,57630.0,NaN,77.7176,77.6089,0.0,0.0,...,5.5000,973.0,failed,1,127.0,MODEL_2,CNTRY_1,127.0,221.0,ENG_2


## 2 · Auditoría de esquema y calidad

### 2.1 · Cobertura del join

La pregunta que bloquea todo lo demás: ¿los mismos vehículos están en las tres
tablas? F1 dijo que sí (riesgo #1 del plan §10 descartado). Acá se re-verifica
sobre los códigos **ya canonizados**, que es como van a llegar al panel.

In [5]:
join_coverage(static, scans)

,check,n
0,vehículos en `vehicles`,1081
1,vehículos en `trips`,1081
2,vehículos en `signals`,1081
3,vehículos en las tres tablas,1081
4,en `vehicles` y no en `trips`,0
5,en `trips` y no en `vehicles`,0
6,en `vehicles` y no en `signals`,0
7,en `signals` y no en `vehicles`,0


### 2.2 · Nulos y tipos por columna

Sobre las tres tablas completas. Los nulos que importan para F2 están en
[`f1-calidad-odometro.md`](../docs/memoria/f1-calidad-odometro.md); acá se
contrastan contra lo documentado.

In [6]:
quality = quality_report(scans)
quality[quality["n_null"] > 0].sort_values(["table", "null_frac"], ascending=[True, False])

,table,column,dtype,n_null,null_frac
39,signals,Regenerations,str,10561841,0.994728
40,signals,DistanceBetweenRegenerations,float64,10561841,0.994728
36,signals,OdometerValue,float64,1183603,0.111473
35,signals,eventTimestamp,"datetime64[us, UTC]",18124,0.001707
38,signals,Message,str,4227,0.000398
13,trips,KilometerPerHour,float64,821929,0.324689
16,trips,EngineOilLifePCStart,float64,6754,0.002668
17,trips,EngineOilLifePCEnd,float64,6754,0.002668
21,trips,AirFilterStart,str,6754,0.002668
23,trips,AirRegenerationStart,float64,6754,0.002668


In [7]:
# Columnas sin ningún nulo, por tabla (el resto de la foto).
quality[quality["n_null"] == 0].groupby("table")["column"].agg(["size", list])

,size,list
table,,
signals,3,"[VehicleCode, Acumulation, cohort]"
trips,13,"[VehicleCode, EngineTemperatureMin, EngineTemp..."
vehicles,6,"[VehicleCode, ProductionDay, Engine, ModelSeri..."


### 2.3 · Duplicados

Tres capas distintas, y conviene no confundirlas:

- `dup_mismo_codigo`: la misma fila publicada bajo el mismo `VehicleCode` en las
  dos cohortes.
- `dup_clones`: lo que colapsa el dedupe de los 13 vehículos que vienen con dos
  códigos ([`f1-clones-vehiculos.md`](../docs/memoria/f1-clones-vehiculos.md)).
- `dup_otros`: filas exactamente repetidas **dentro de un mismo archivo**, que el
  dedupe de vehículos no toca porque no son un problema de identidad de vehículo.

In [8]:
duplicate_report(scans)

,table,n_rows,dup_mismo_codigo,dup_total_canonico,dup_clones,dup_otros,dup_otros_frac
0,vehicles,1120,0,26,26,0,0.000000
1,trips,2531435,62064,93096,93096,0,0.000000
2,signals,10617817,378723,505252,384867,120385,0.011338


### 2.4 · Re-confirmación de los hallazgos de F1

No se re-descubre nada desde cero: se contrasta contra lo que ya está documentado,
y si algo no coincide se marca en vez de asumir que el doc viejo tiene razón.

**(a) Sesgo de `ENG_3`** ([`f1-sesgo-eng3.md`](../docs/memoria/f1-sesgo-eng3.md)):
el doc reporta 268 vehículos sanos `ENG_3` y **cero** con evento.

In [9]:
engine = pd.crosstab(vehicles["static_Engine"], vehicles["event_observed"])
engine.columns = ["sanos", "con_evento"]
engine.assign(
    frac_sanos=lambda d: d["sanos"] / d["sanos"].sum(),
    frac_con_evento=lambda d: d["con_evento"] / d["con_evento"].sum(),
).round(4)

,sanos,con_evento,frac_sanos,frac_con_evento
static_Engine,,,,
ENG_1,125,72,0.1746,0.1973
ENG_2,323,293,0.4511,0.8027
ENG_3,268,0,0.3743,0.0000


**(b) `IdentificationDate` como etiqueta de cohorte.** Esta verificación tiene que
hacerse sobre la tabla **cruda**, no sobre la deduplicada: `dedupe_vehicles()`
*deriva* `cohort` de `IdentificationDate`, así que post-dedupe la equivalencia se
cumple por construcción y el chequeo no probaría nada. En el crudo, en cambio,
`cohort` viene del archivo del que salió la fila.

In [10]:
raw_vehicles = load_table("vehicles", cfg["sources"])
pd.crosstab(raw_vehicles["cohort"], raw_vehicles["IdentificationDate"].notna()).rename(
    columns={False: "IdentificationDate nula", True: "IdentificationDate presente"}
)

IdentificationDate,IdentificationDate nula,IdentificationDate presente
cohort,,
failed,0,378
not_failed,742,0


**(c) Anclaje temporal**
([`f1-anclaje-temporal.md`](../docs/memoria/f1-anclaje-temporal.md)): si
`ProductionDay` está en el mismo eje que el calendario, `primer_viaje −
ProductionDay` tiene que ser casi constante entre vehículos. El doc reporta
std 0,7 d e **IQR 0 d** contra un rango de `ProductionDay` de 338 días, y std
60,6 d para la hipótesis alternativa que suma `daysUntilSale`.

In [11]:
offset = anchor_offset_days(vehicles).dropna()
alternativa = offset - vehicles.loc[offset.index, "static_daysUntilSale"]

pd.DataFrame(
    {
        "primer_viaje - ProductionDay": [offset.std(), offset.quantile(0.75) - offset.quantile(0.25), offset.max() - offset.min()],
        "... - daysUntilSale": [alternativa.std(), alternativa.quantile(0.75) - alternativa.quantile(0.25), alternativa.max() - alternativa.min()],
    },
    index=["std [d]", "IQR [d]", "rango [d]"],
).round(2)

,primer_viaje - ProductionDay,... - daysUntilSale
std [d],0.7,61.97
IQR [d],0.0,73.00
rango [d],12.0,353.00


## 3 · Lo que entra al split

Una fila por `vehicle_id`, con `event_observed` derivado de `IdentificationDate` y
las `static_*` del contrato. Dos aclaraciones que no son cosméticas:

- **`static_Engine` está acá para reportar, no para modelar.** El set base la
  excluye (`features.static_excluded` en `configs/data/panel_v1.yaml`): `ENG_3` es
  el 37% de los sanos y el 0% de los que tienen evento, así que como feature es una
  regla que acierta siempre adentro de este dataset y nunca afuera.
- **`event_day_since_production` (la `IdentificationDate` cruda) no es una
  feature.** Es la etiqueta en días desde producción; se conserva porque F2 la
  necesita para traducir el evento al eje de km, nunca para entrenar.

In [12]:
columnas_split = ["vehicle_id", "event_observed"] + [c for c in vehicles.columns if c.startswith("static_")]
print(columnas_split)
print("tasa de eventos global:", round(vehicles["event_observed"].mean(), 4))
vehicles[columnas_split].head(3)

['vehicle_id', 'event_observed', 'static_ModelSeries', 'static_SalesCountry_cd', 'static_daysUntilSale', 'static_ProductionDay', 'static_Engine']
tasa de eventos global: 0.3377


,vehicle_id,event_observed,static_ModelSeries,static_SalesCountry_cd,static_daysUntilSale,static_ProductionDay,static_Engine
0,VEH_0001,0,MODEL_4,CNTRY_1,47.0,59.0,ENG_1
1,VEH_0002,0,MODEL_4,CNTRY_1,45.0,54.0,ENG_1
2,VEH_0003,0,MODEL_4,CNTRY_1,110.0,92.0,ENG_1


## 4 · Holdout dev/test 80/20

La lógica está en `src/eval/splits.py::make_test_split` —misma casa que la CV, y
por el mismo motivo: si cada notebook arma su propio split, tarde o temprano uno
se olvida de agrupar por vehículo. Acá solo se ejecuta.

`make_test_split` colapsa a nivel vehículo, estratifica por `event_observed` y usa
el mismo `StratifiedGroupKFold` que `make_splits`, quedándose con un fold como
test (con `test_size=0.2`, 5 folds → el fold 0).

Si el holdout ya está congelado, el notebook **no lo pisa**: lo carga y verifica
que coincide. Misma semilla, mismo split; si algún día no coincidiera, es que
cambió el universo de vehículos y hay que enterarse ahí, no tres semanas después.

In [13]:
split = make_test_split(
    vehicles,
    test_size=float(cfg["test_size"]),
    seed=int(cfg["seed"]),
    group_column=cfg["group_column"],
    event_column=cfg["event_column"],
)

frozen_path = resolve_path(cfg["output"]["test_split"])
if frozen_path.exists():
    frozen = load_test_split(frozen_path)
    coincide = frozen["test_vehicles"] == split["test_vehicles"]
    print(f"holdout congelado el {frozen['created_at']} · reproduce igual: {coincide}")
    assert coincide, "El split no reproduce el archivo congelado: cambió el universo de vehículos"
    split = frozen
else:
    save_test_split(split, frozen_path)
    print(f"holdout congelado en {frozen_path}")

{k: v for k, v in split.items() if k not in ("test_vehicles", "dev_vehicles")}

holdout congelado el 2026-09-16T03:10:51+00:00 · reproduce igual: True


{'created_at': '2026-09-16T03:10:51+00:00',
 'kind': 'dev_test_holdout',
 'test_size_requested': 0.2,
 'test_size_achieved': 0.20074005550416282,
 'equivalent_folds': 5,
 'seed': 42,
 'group_column': 'vehicle_id',
 'event_column': 'event_observed',
 'panel': {'n_rows': 1081,
  'n_vehicles': 1081,
  'vehicles_sha256_16': 'f6c9b84d40d83acd'},
 'n_vehicles': 1081,
 'n_event_vehicles': 365,
 'dev': {'n_vehicles': 864,
  'n_event_vehicles': 292,
  'event_rate': 0.33796296296296297},
 'test': {'n_vehicles': 217,
  'n_event_vehicles': 73,
  'event_rate': 0.33640552995391704}}

## 5 · Validación del split

### 5.1 · Tasa de eventos y regla 2

La tasa de eventos tiene que quedar casi idéntica entre dev y test (para eso está
la estratificación), y ningún vehículo puede aparecer de los dos lados.

In [14]:
resumen = pd.DataFrame([split["dev"], split["test"]], index=["dev", "test"])
resumen["frac_del_universo"] = resumen["n_vehicles"] / split["n_vehicles"]
resumen.round(4)

,n_vehicles,n_event_vehicles,event_rate,frac_del_universo
dev,864,292,0.3380,0.7993
test,217,73,0.3364,0.2007


In [15]:
dev_mask, test_mask = test_split_masks(vehicles, split)
solapan = set(vehicles.loc[dev_mask, "vehicle_id"]) & set(vehicles.loc[test_mask, "vehicle_id"])
print("vehículos en los dos lados:", len(solapan))
print("vehículos cubiertos:", int(dev_mask.sum() + test_mask.sum()), "de", len(vehicles))
print("delta de tasa de eventos (test - dev):", round(split["test"]["event_rate"] - split["dev"]["event_rate"], 4))

vehículos en los dos lados: 0
vehículos cubiertos: 1081 de 1081
delta de tasa de eventos (test - dev): -0.0016


### 5.2 · Cómo quedaron repartidas las estáticas

Esto es un **reporte, no una restricción**: el split se estratifica solo por
evento. Con 1081 vehículos, forzar además el balance de motor, modelo y país gasta
grados de libertad para arreglar algo que la estratificación por evento ya deja
razonable. Lo que sí hay que hacer es mirarlo: si una categoría quedara casi
entera de un lado, el número del test mediría esa categoría y no el modelo.

In [16]:
balance = split_balance(vehicles, split, list(cfg["balance_columns"]))
balance.sort_values("diff", key=abs, ascending=False).round(4)

,variable,value,n_dev,n_test,frac_dev,frac_test,diff
5,static_ModelSeries,MODEL_3,107,36,0.1238,0.1659,0.0421
9,static_SalesCountry_cd,CNTRY_3,123,23,0.1424,0.1060,-0.0364
3,static_ModelSeries,MODEL_1,234,52,0.2708,0.2396,-0.0312
10,static_SalesCountry_cd,CNTRY_4,200,57,0.2315,0.2627,0.0312
7,static_SalesCountry_cd,CNTRY_1,255,70,0.2951,0.3226,0.0274
1,static_Engine,ENG_2,496,120,0.5741,0.5530,-0.0211
11,static_SalesCountry_cd,CNTRY_5,42,7,0.0486,0.0323,-0.0164
6,static_ModelSeries,MODEL_4,229,54,0.2650,0.2488,-0.0162
2,static_Engine,ENG_3,212,56,0.2454,0.2581,0.0127
0,static_Engine,ENG_1,156,41,0.1806,0.1889,0.0084


In [17]:
# La categoría más desbalanceada, en puntos porcentuales.
peor = balance.loc[balance["diff"].abs().idxmax()]
print(f"máximo desvío: {peor['variable']}={peor['value']} · dev {peor['frac_dev']:.3f} vs test {peor['frac_test']:.3f} ({peor['diff'] * 100:+.1f} pp)")

máximo desvío: static_ModelSeries=MODEL_3 · dev 0.124 vs test 0.166 (+4.2 pp)


## 6 · Qué queda congelado

- `data/processed/test_split.json` — las dos listas de `vehicle_id` con su
  metadata (semilla, `test_size`, fecha, huella del universo). **No se regenera**:
  cambiar la semilla después de haber medido contra el test es elegir el test que
  mejor queda.
- `data/processed/vehicles.parquet` — la unión a nivel vehículo, para que F2 no
  vuelva a leer 1,2 GB.
- `data/processed/raw_quality.csv` — nulos y tipos por columna.

Nada de eso se versiona (`data/` está en `.gitignore`): se regenera con el script
o se comparte como wandb Artifact.

**Lo que sigue (F2):** construir el panel con los 1081 vehículos y entrenar solo
sobre `dev_mask`. Las filas de test quedan escritas pero nadie las mira hasta
tener el modelo elegido: así la evaluación final no obliga a regenerar el panel, y
la protección la da el código en vez de la ausencia del archivo. Dos entradas ya
listas:

- `iter_enriched_table("trips" | "signals", static)` — las tablas grandes por
  chunks, con ids canónicos, clones colapsados y estáticas pegadas.
- `test_split_masks(panel, split)` — aplica el holdout congelado sobre el panel.
  Falla si el panel trae vehículos que el holdout no conoce, que es exactamente el
  caso en el que un vehículo se colaría a dev sin que nadie lo note.

Los hallazgos y las desviaciones contra F1 quedaron escritos en
[`docs/memoria/f2-union-y-holdout-dev-test.md`](../docs/memoria/f2-union-y-holdout-dev-test.md),
y el motivo del 80/20 antes de F2 en
[`docs/memoria/decisiones.md`](../docs/memoria/decisiones.md).